## 환경 준비

아래 셀은 이 노트북에 필요한 Python 패키지가 설치되어 있는지 확인하고,
없으면 자동으로 설치한다. 이미 설치되어 있으면 빠르게 스킵된다.
터미널에서 미리 `uv sync`를 했다면 이 셀은 아무것도 설치하지 않는다.


In [ ]:
# === 의존성 자동 설치 (이미 설치되어 있으면 빠르게 스킵됨) ===
import subprocess, sys

_IMPORT_NAME_OVERRIDES = {
    "scikit-learn": "sklearn",
    "python-dateutil": "dateutil",
    "beautifulsoup4": "bs4",
}


def _ensure_packages(*packages):
    """누락된 패키지만 설치. 이미 있으면 스킵."""
    missing = []
    for pkg in packages:
        name = pkg.split(">=")[0].split("==")[0].split("[")[0]
        import_name = _IMPORT_NAME_OVERRIDES.get(name, name.replace("-", "_"))
        try:
            __import__(import_name)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("All packages already installed ✓")

_ensure_packages(
    "pandas", "numpy", "matplotlib", "scikit-learn",
    "pydantic", "python-dateutil", "rich", "tqdm",
)


# 에이전트 워크플로우(Agentic Workflow)

이 노트북은 baseline RAG를 stateful workflow로 확장한 이유를 보여준다. 핵심은 질문을 곧바로 답하지 않고, 상태(state)를 만들고, 노드(node)별로 상태를 갱신하며, trace를 남기고, 마지막에 근거 검증(grounding verification)과 fallback까지 수행하는 것이다.

## 학습 목표
- 9개 workflow node가 각각 어떤 입력을 받고 어떤 출력을 만드는지 설명할 수 있다.
- `AgentState`가 단순 chain과 어떻게 다른지, 왜 디버깅과 재현성에 유리한지 이해한다.
- trace의 `timestamp`, `latency`, `inputs`, `outputs` 컬럼을 읽을 수 있다.
- `use_llm=True` 경로가 존재하지만 기본은 규칙 기반이며, LLM 통합은 09번 노트북에서 자세히 다룬다는 연결을 이해한다.


## 개념 설명

단순 chain은 "입력 → 검색 → 답변"처럼 직선형으로 흐르기 쉽다. 반면 stateful workflow는 각 단계가 상태를 읽고 쓰며, 중간 산출물을 다음 단계가 재사용한다. 이 차이가 중요한 이유는, 잘못된 답변이 나왔을 때 어디에서 문제가 생겼는지 node 단위로 거슬러 올라갈 수 있기 때문이다.

이 저장소의 workflow는 다음 순서를 유지한다. `normalize_query -> classify_query -> make_plan -> retrieve_docs -> decide_tools -> run_tools -> synthesize_answer -> verify_grounding -> fallback_or_finalize`.

**목적**
- baseline과 달리 왜 상태 기반 설계를 쓰는지 먼저 이해한다.

**핵심 로직**
- 각 node가 상태 일부를 갱신한다.
- 각 node가 trace를 남겨 중간값을 inspectable하게 만든다.
- 검증과 fallback이 마지막에 붙어 답변 품질과 안전성을 동시에 관리한다.

**결과 해석 가이드**
- 이 notebook의 표는 단순 성능이 아니라 "workflow가 제대로 분해되었는가"를 읽는 용도다.

**💡 면접 포인트**
- "Stateful workflow는 중간 산출물을 명시적으로 남겨서 디버깅, 평가, 회귀 검증이 쉬워진다"고 설명하면 좋다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## Agentic workflow란 무엇인가

먼저 환경과 import를 준비한다. 여기서 중요한 점은, notebook 안에서 핵심 로직을 재정의하지 않고 모두 `src/`에서 가져온다는 것이다. 교육용 notebook일수록 코드와 설명이 분리돼 있어야, 실험이 아닌 재사용 가능한 시스템으로 설명할 수 있다.

**목적**
- 실습에서 사용할 retriever, state helper, trace viewer, node 함수를 한 번에 준비한다.

**핵심 로직**
- `build_demo_index()`로 corpus와 retriever를 초기화한다.
- `create_initial_state`, `display_trace`, 각 node 함수는 workflow의 개별 조각을 직접 실행해 보기 위한 진입점이다.
- `run_workflow()`는 이 조각들을 한 번에 실행하는 전체 엔트리포인트다.

**주요 파라미터/변수**
- `retriever`: 모든 retrieval node가 공유하는 검색기
- `run_workflow`: 전체 9노드를 실행하는 함수
- `use_llm=True`: LLM synthesizer/verifier 경로를 켤 수 있지만 상세 해설은 09번 notebook에서 다룬다.


## 구현 준비

아래 셀이 준비하는 것 중 가장 중요한 것은 `AgentState`와 node 함수들이다. 이 notebook은 전체 workflow를 한 번에 보여주지 않고, 상태를 조금씩 바꾸며 "왜 이 node가 필요한가"를 단계별로 확인한다.

**목적**
- 실험에 필요한 객체를 명확히 분리해 둔다.

**핵심 로직**
- retriever는 근거 검색을 담당한다.
- state helper는 상태 초기화와 schema inspection을 담당한다.
- trace helper는 나중에 노드별 입출력을 테이블로 보여준다.

**결과 해석 가이드**
- 이 셀은 표를 만들지 않지만, 이후 모든 셀이 공유하는 실험 기반을 세팅한다.


In [ ]:
import pandas as pd

from src.ingestion import build_demo_index
from src.state import AgentState, create_initial_state
from src.trace_debug import display_trace
from src.workflow import (
    classify_query_node,
    decide_tools_node,
    fallback_or_finalize_node,
    make_plan_node,
    normalize_query_node,
    retrieve_docs_node,
    run_tools_node,
    run_workflow,
    synthesize_answer_node,
    verify_grounding_node,
)

retriever = build_demo_index(persist=False)

## 상태 기반 agent 설계(Stateful agent design)

아래 셀은 `AgentState`의 필드 목록을 표로 보여준다. 이 표는 단순한 타입 목록이 아니라, workflow가 어떤 중간 산출물을 명시적으로 관리하는지 보여주는 계약(contract)이다. baseline과 가장 크게 다른 지점이 바로 여기에 있다.

**목적**
- workflow가 어떤 상태를 들고 다니는지 명시적으로 확인한다.

**핵심 로직**
- `AgentState`는 질의, 계획, 검색 결과, 도구 요청/출력, 초안 답변, 검증 결과, 최종 상태, trace를 한 구조에 모은다.
- 이 구조 덕분에 evaluator와 debugger가 같은 상태를 그대로 읽을 수 있다.

**주요 파라미터/변수**
- `normalized_query`: 정규화된 질문
- `retrieved_docs`: 검색된 근거 청크
- `verification_result`: verifier가 만든 구조화된 판정
- `trace`: node별 실행 흔적

**실제 소스 코드: AgentState TypedDict — src/state.py**
```python
class AgentState(TypedDict, total=False):
    session_id: str
    user_query: str
    normalized_query: str
    query_type: str
    requires_tools: bool
    plan: list[str]
    retrieved_docs: list[dict[str, Any]]
    retrieved_memories: list[dict[str, Any]]
    tool_requests: list[dict[str, Any]]
    tool_outputs: list[dict[str, Any]]
    draft_answer: str
    citations: list[dict[str, Any]]
    verification_result: VerificationResult
    final_answer: str
    final_status: str
    trace: list[dict[str, Any]]
    memory_updates: list[dict[str, Any]]
    errors: list[str]
```

**코드 읽기 포인트**
- `TypedDict(total=False)`라서 상태가 단계적으로 채워질 수 있다.
- 초기에는 없는 키도 workflow가 진행되며 차례대로 생긴다.
- 메모리 확장 경로를 위해 `retrieved_memories`, `memory_updates`도 이미 포함돼 있다.

**결과 해석 가이드**
- 표에서 필드가 많아 보이는 것은 복잡함이 아니라 observability다. "무슨 일이 일어났는지 모르는 chain"보다 훨씬 디버깅하기 쉽다.


In [ ]:
state_schema = pd.DataFrame(
    {
        'field': list(AgentState.__annotations__.keys()),
        'type': [str(value) for value in AgentState.__annotations__.values()],
    }
)
state_schema

## 노드(node) 개념

이제 실제 상태 객체를 만든 뒤 첫 번째 node를 실행한다. node 설계의 핵심은 "상태를 조금 바꾸고, 그 변화 자체를 trace에 남긴다"는 원칙이다. 이 원칙이 있어야 later-stage bug가 생겨도 upstream 상태를 재구성할 수 있다.

**목적**
- 초기 상태 생성과 trace 기록 방식이 어떻게 연결되는지 이해한다.

**핵심 로직**
- `create_initial_state()`는 session_id, user_query, 빈 trace, 빈 errors를 준비한다.
- `normalize_query_node()`는 사용자 질문을 정규화하고 trace를 남긴다.
- `append_trace()`와 `_record_node_trace()`가 trace의 공통 형식을 강제한다.

**주요 파라미터/변수**
- `session_id`: 한 실행(run)을 구분하는 ID
- `inputs/outputs`: 각 node가 받은 값과 만든 값
- `latency`: 노드별 실행 시간

**실제 소스 코드: create_initial_state() — src/state.py**
```python
def create_initial_state(user_query: str) -> AgentState:
    return AgentState(
        session_id=str(uuid.uuid4()),
        user_query=user_query,
        trace=[],
        errors=[],
    )
```

**실제 소스 코드: append_trace() — src/state.py**
```python
def append_trace(state: AgentState, node: str, payload: dict[str, Any]) -> None:
    inputs = payload.get("inputs", {}) if isinstance(payload, dict) else {}
    outputs = payload.get("outputs", payload) if isinstance(payload, dict) else payload
    latency = payload.get("latency") if isinstance(payload, dict) else None
    state.setdefault("trace", []).append(
        {
            "node": node,
            "timestamp": iso_timestamp(),
            "payload": json_ready(payload),
            "inputs": json_ready(inputs),
            "outputs": json_ready(outputs),
            "latency": round(float(latency), 6) if isinstance(latency, (int, float)) else None,
        }
    )
```

**실제 소스 코드: _record_node_trace() — src/workflow.py**
```python
def _record_node_trace(
    state: AgentState,
    node_name: str,
    inputs: dict[str, Any],
    outputs: dict[str, Any],
    start_time: float,
) -> None:
    append_trace(
        state,
        node_name,
        {
            "inputs": inputs,
            "outputs": outputs,
            "latency": round(time.perf_counter() - start_time, 6),
        },
    )
    validate_state(state)
```

**코드 읽기 포인트**
- `create_initial_state()`는 처음부터 모든 필드를 채우지 않고, 필수 최소값만 만든다.
- `append_trace()`는 payload에서 `inputs`, `outputs`, `latency`를 꺼내 trace entry 표준 형식으로 저장한다.
- `_record_node_trace()`는 append 후 바로 `validate_state()`를 호출해, 노드가 끝난 시점 상태가 계약에 맞는지 검사한다.
- 즉 trace는 단순 로그가 아니라, 관측 가능성과 상태 일관성 검사를 같이 묶은 장치다.

**결과 해석 가이드**
- 이 셀 출력에서 `user_query`와 `normalized_query`가 다르면 정규화가 의미 있게 작동한 것이다.


In [ ]:
state = create_initial_state('How many days are in the pilot window?')
normalize_query_node(state)
pd.Series({'user_query': state['user_query'], 'normalized_query': state['normalized_query']})

## 질의 분류(query classification)

agentic workflow의 첫 번째 지능형 분기는 질문 유형을 맞히는 것이다. 같은 retrieval을 하더라도, summary 질문과 multi-hop 질문은 이후 계획과 도구 사용이 달라진다. 그래서 classifier는 단순하지만 전체 workflow 방향을 결정하는 고레버리지(high-leverage) node다.

**목적**
- 질문을 `simple_lookup`, `comparison`, `multi_hop`, `summary`, `insufficient_evidence_risk` 중 하나로 나눈다.

**핵심 로직**
- 질문을 소문자/정규화한 뒤 marker 집합을 차례대로 검사한다.
- `insufficient_evidence_risk`를 먼저 판정해 문서 밖 질문을 보수적으로 처리한다.
- `requires_tools`도 함께 반환해서 이후 tool planning과 연결한다.

**주요 파라미터/변수**
- `SUMMARY_MARKERS`: 요약 의도 감지
- `COMPARISON_MARKERS`: 비교 질문 감지
- `MULTI_HOP_MARKERS`: 계산·이유·영향 같은 다단계 질문 감지
- `TOOL_MARKERS`: 도구가 필요한 비교/계산 신호

**실제 소스 코드: classify_query() — src/classifier.py**
```python
def classify_query(query: str) -> dict[str, object]:
    normalized = normalize_text(query).lower()

    if any(marker in normalized for marker in INSUFFICIENT_MARKERS):
        return {"query_type": "insufficient_evidence_risk", "requires_tools": False}

    if any(marker in normalized for marker in SUMMARY_MARKERS):
        return {"query_type": "summary", "requires_tools": False}

    if any(marker in normalized for marker in COMPARISON_MARKERS):
        requires_tools = any(marker in normalized for marker in TOOL_MARKERS)
        return {"query_type": "comparison", "requires_tools": requires_tools}

    if any(marker in normalized for marker in MULTI_HOP_MARKERS):
        return {"query_type": "multi_hop", "requires_tools": True}

    return {"query_type": "simple_lookup", "requires_tools": False}
```

**코드 읽기 포인트**
- marker 기반 분류기라서 explainability는 높고, recall은 제한적이다.
- `requires_tools=True`는 "나중에 툴 후보를 검토하라"는 신호이지, 곧바로 툴을 실행한다는 뜻은 아니다.
- 문서 밖 질문을 초기에 감지하면 verifier와 fallback 부담을 줄일 수 있다.

**결과 해석 가이드**
- `query_type`이 예상과 다르면 이후 전체 workflow가 빗나갈 수 있으므로, 이 단계는 trace에서 가장 먼저 확인할 후보 중 하나다.

**💡 면접 포인트**
- "Classifier는 단순 룰 기반이라도 downstream 비용을 크게 바꾼다. 잘못 분류하면 plan, tool, fallback이 전부 연쇄적으로 흔들린다"고 설명할 수 있다.


In [ ]:
classify_query_node(state)
pd.Series({'query_type': state['query_type'], 'requires_tools': state['requires_tools']})

## 계획(planning)

분류가 끝나면 agent는 바로 답을 쓰지 않고, 질문 유형에 맞는 reasoning template을 고른다. 이 저장소의 planner는 LLM planner가 아니라 template planner이지만, 바로 그 점 때문에 어떤 질문에 어떤 절차를 강제하는지 투명하게 읽을 수 있다.

**목적**
- 질문 유형별로 실행 절차의 뼈대를 만든다.

**핵심 로직**
- `PLAN_TEMPLATES`에 query type별 step list가 들어 있다.
- `make_plan()`은 선택된 템플릿을 복사해 반환한다.
- 이후 trace에서 어떤 plan이 선택되었는지 바로 확인할 수 있다.

**주요 파라미터/변수**
- `query_type`: 분류 결과
- `plan`: ordered list of execution steps

**실제 소스 코드: PLAN_TEMPLATES + make_plan() — src/planner.py**
```python
PLAN_TEMPLATES = {
    "simple_lookup": [
        "retrieve relevant evidence",
        "draft concise grounded answer",
        "verify citation support",
    ],
    "comparison": [
        "retrieve evidence for each target",
        "compare aligned facts",
        "draft structured comparison answer",
        "verify grounding",
    ],
    "multi_hop": [
        "decompose question into sub-parts",
        "retrieve multi-source evidence",
        "optionally call helper tools",
        "synthesize reasoning chain",
        "verify grounding before final answer",
    ],
    "summary": [
        "retrieve major sections",
        "compress key points",
        "verify key claims are grounded",
    ],
    "insufficient_evidence_risk": [
        "retrieve the closest available evidence",
        "check whether the request exceeds document scope",
        "abstain if grounding is weak",
    ],
}


def make_plan(query_type: str) -> list[str]:
    return PLAN_TEMPLATES.get(query_type, PLAN_TEMPLATES["simple_lookup"]).copy()
```

**코드 읽기 포인트**
- template 자체는 짧지만, "검색 먼저"인지 "도구 사용 포함"인지가 명확히 드러난다.
- `list(template)`로 복사해서 원본 템플릿이 오염되지 않게 한다.
- 이 planner는 deterministic해서 평가 실험에서 재현성이 높다.

**결과 해석 가이드**
- 표에서 `planned_step` 순서를 보면 workflow가 어떤 reasoning skeleton을 택했는지 읽을 수 있다.


In [ ]:
make_plan_node(state)
pd.DataFrame({'planned_step': state['plan']})

## 검색(retrieval)

retrieval node는 normalized query와 plan을 받아 실제 근거 청크를 찾아온다. 이 단계는 baseline에서도 있었지만, agent workflow에서는 이후의 tool selection, synthesis, verification이 모두 이 결과를 공유한다는 점이 다르다.

**목적**
- 질문을 실제 근거 청크 집합으로 바꾼다.

**핵심 로직**
- `retrieve_docs_node()`는 retriever 인터페이스의 `search()`만 호출하므로 backend 교체가 쉽다.
- 검색 결과 수와 source 목록을 trace output에 남겨, 나중에 retrieval miss를 빠르게 진단할 수 있다.

**주요 파라미터/변수**
- `top_k=4`: 이번 실습에서 상위 4개 근거만 본다.
- `score`: retriever가 계산한 관련도 점수

**결과 해석 가이드**
- `score` 상위 청크의 `source`가 질문과 맞지 않으면, 이후 node가 아무리 잘해도 답이 흔들린다.
- retrieval 결과는 "답변"이 아니라 "근거 후보"라는 점을 계속 구분해 읽어야 한다.


In [ ]:
retrieve_docs_node(state, retriever=retriever, top_k=4)
pd.DataFrame(state['retrieved_docs'])[['chunk_id', 'source', 'score', 'text']]

## 도구(tools)

모든 질문이 도구를 필요로 하지는 않지만, 날짜 차이 계산처럼 문서를 읽는 것만으로는 바로 답하기 어려운 질문은 별도 툴이 필요하다. agentic workflow의 장점은 도구 사용 여부를 명시적으로 결정하고, 그 요청과 결과를 상태에 남긴다는 점이다.

**목적**
- 검색 결과와 질문 유형을 바탕으로 어떤 툴을 쓸지 결정한다.

**핵심 로직**
- `plan_tool_requests()`는 질문과 evidence text를 보고 `ToolRequest` 리스트를 만든다.
- 날짜 차이 질문이면 `date_parser` 두 번과 `calculator` 한 번을 예약한다.
- summary 질문이면 `keyword_extractor`를 예약해 synthesis에 보조 신호를 준다.

**주요 파라미터/변수**
- `query`: 도구 필요성을 판단할 질문
- `query_type`: summary인지 multi-hop인지 등 분류 결과
- `retrieved_docs`: evidence text에서 날짜 후보를 뽑기 위한 입력

**실제 소스 코드: plan_tool_requests() — src/tools.py**
```python
def plan_tool_requests(
    query: str,
    query_type: str,
    retrieved_docs: list[dict[str, Any]],
) -> list[ToolRequest]:
    requests: list[ToolRequest] = []
    normalized = normalize_text(query).lower()
    evidence_text = " ".join(doc["text"] for doc in retrieved_docs)
    date_candidates = extract_dates(query) or extract_dates(evidence_text)

    if "how many days" in normalized or "days" in normalized or "며칠" in normalized:
        if len(date_candidates) >= 2:
            requests.append(ToolRequest("date_parser", {"text": date_candidates[0]}))
            requests.append(ToolRequest("date_parser", {"text": date_candidates[1]}))
            requests.append(ToolRequest("calculator", {"expression": "DATE_DIFF_DAYS"}))

    if query_type == "summary":
        requests.append(ToolRequest("keyword_extractor", {"text": evidence_text or query, "limit": 5}))

    return requests
```

**코드 읽기 포인트**
- `date_candidates = extract_dates(query) or extract_dates(evidence_text)`: 질문 자체에 날짜가 없으면 검색된 문서에서 날짜를 찾는다.
- `DATE_DIFF_DAYS`는 calculator에 바로 수식을 주는 대신, 나중에 parsed date를 이용해 계산하라는 예약 신호다.
- 툴 계획과 실제 실행을 분리해 두면, 디버깅할 때 "선택이 잘못됐는가"와 "실행이 실패했는가"를 나눠 볼 수 있다.

**결과 해석 가이드**
- `tool_outputs`가 비어 있으면 이 질문은 도구 없이도 답할 수 있다고 workflow가 판단한 것이다.
- 예상과 다르게 도구가 빠졌다면 classifier와 planner, tool request planning을 같이 점검해야 한다.


In [ ]:
decide_tools_node(state)
run_tools_node(state)

pd.DataFrame(state['tool_outputs']) if state['tool_outputs'] else pd.DataFrame([{'message': 'No tool outputs'}])

## 답변 합성(answer synthesis)

synthesis node는 retrieved docs와 tool outputs를 받아 사람이 읽을 문장으로 묶는다. baseline과 다른 점은, query type에 따라 문장 선택 방식이 달라지고, tool output이 있으면 답변 본문에 반영된다는 것이다. 하지만 아직 이 단계만으로는 hallucination을 막을 수 없다. 그래서 바로 다음 verifier가 필요하다.

**목적**
- 검색 근거와 도구 결과를 묶어 draft answer를 만든다.

**핵심 로직**
- 질문 관련성이 높은 문장을 `_rank_sentences()`로 다시 정렬한다.
- query type별로 비교/요약/날짜/다중근거 문장 조합 전략이 다르다.
- `calculator`나 `keyword_extractor` 결과가 있으면 보조 설명으로 붙인다.

**주요 파라미터/변수**
- `query_type`: 어떤 답변 템플릿을 고를지 결정
- `tool_outputs`: 계산값이나 키워드처럼 문장 조합에 들어갈 보조 결과
- `citations`: 상위 근거 청크 메타데이터

**실제 소스 코드: synthesize_answer() — src/synthesizer.py**
```python
def synthesize_answer(
    query: str,
    query_type: str,
    retrieved_docs: list[dict[str, Any]],
    tool_outputs: list[dict[str, Any]],
) -> dict[str, Any]:
    normalized_query = normalize_text(query).lower()
    ranked = _rank_sentences(query, retrieved_docs)
    evidence_sentences = [sentence for _, sentence, _ in ranked[:3]]
    citations = []
    seen_chunks: set[str] = set()
    for _, _, doc in ranked[:3]:
        if doc["chunk_id"] in seen_chunks:
            continue
        citations.append(
            {
                "doc_id": doc["doc_id"],
                "chunk_id": doc["chunk_id"],
                "source": doc["source"],
                "score": doc["score"],
            }
        )
        seen_chunks.add(doc["chunk_id"])

    if not evidence_sentences:
        draft = "I could not find grounded evidence for this question in the loaded documents."
        return {"draft_answer": draft, "citations": citations}

    calculator_result = next(
        (
            output["output"]["result"]
            for output in tool_outputs
            if output["tool_name"] == "calculator" and output["output"].get("ok")
        ),
        None,
    )
    date_mentions = extract_dates(" ".join(doc["text"] for doc in retrieved_docs))

    if ("how many days" in normalized_query or "days" in normalized_query) and calculator_result is not None:
        if len(date_mentions) >= 2:
            body = f"The pilot window spans {calculator_result} days, from {date_mentions[0]} to {date_mentions[1]}."
        else:
            body = f"The relevant duration is {calculator_result} days."
    elif query_type == "comparison":
        comparison_sentence = next(
            (sentence for sentence in evidence_sentences if "while" in sentence.lower() or "different" in sentence.lower()),
            evidence_sentences[0],
        )
        body = comparison_sentence
    elif query_type == "summary":
        distinct_sentences: list[str] = []
        seen_sources: set[str] = set()
        for _, sentence, doc in ranked:
            if doc["source"] in seen_sources:
                continue
            distinct_sentences.append(sentence)
            seen_sources.add(doc["source"])
            if len(distinct_sentences) == 3:
                break
        body = " ".join(distinct_sentences)
    elif "risk" in normalized_query:
        body = next((sentence for sentence in evidence_sentences if "risk" in sentence.lower()), evidence_sentences[0])
    elif "goal" in normalized_query:
        body = next((sentence for sentence in evidence_sentences if "goal" in sentence.lower()), evidence_sentences[0])
    elif "when" in normalized_query and date_mentions:
        body = next((sentence for sentence in evidence_sentences if extract_dates(sentence)), evidence_sentences[0])
    elif query_type == "multi_hop":
        body = " ".join(evidence_sentences[:2])
    else:
        body = evidence_sentences[0]

    notes = _tool_notes(tool_outputs)
    if notes:
        body = f"{body} {' '.join(notes)}"

    source_list = ", ".join(citation["source"] for citation in citations) or "no sources"
    draft = f"{body} Sources: {source_list}."
    return {"draft_answer": draft, "citations": citations}
```

**코드 읽기 포인트**
- `calculator_result`가 있으면 "며칠" 같은 질문에 문서 문장만 인용하는 대신 계산 결과를 명시한다.
- `query_type == "summary"`일 때는 source가 다른 문장을 최대 3개 모아 coverage를 넓힌다.
- 마지막에 `Sources: ...`를 붙여 근거 출처를 답변 문자열 안에도 남긴다.
- 더 자연스러운 생성이 필요하면 `use_llm=True`를 켤 수 있지만, 그 LLM 경로는 09번 notebook에서 상세히 다룬다.

**결과 해석 가이드**
- `citation_count`가 0이면 synthesis 단계가 쓸 만한 근거를 거의 못 찾았다는 신호다.
- draft answer가 자연스러워 보여도 아직 grounded라는 뜻은 아니다.


In [ ]:
synthesize_answer_node(state)
pd.Series({'draft_answer': state['draft_answer'], 'citation_count': len(state['citations'])})

## 검증(verification)

verifier는 agent workflow의 신뢰도를 결정하는 핵심 node다. 검색된 근거를 바탕으로 생성한 문장이 실제로 뒷받침되는지 검사하지 않으면, 시스템은 "근거를 썼다"고 말하면서도 사실은 외부 기억이나 추측을 섞을 수 있다. 이 단계가 바로 hallucination을 제어하는 브레이크다.

**목적**
- draft answer가 retrieved docs와 tool outputs에 실제로 근거하는지 판정한다.

**핵심 로직**
- 답변 문장을 문장 단위로 나눈 뒤 각 문장이 evidence와 얼마나 겹치는지 계산한다.
- calculator/date parser 결과는 토큰 겹침이 낮아도 합법적인 근거가 될 수 있으므로 별도 보정한다.
- `coverage_score`와 `unsupported_claims`를 함께 만들어 fallback 판단에 넘긴다.

**주요 파라미터/변수**
- `draft_answer`: 검증 대상 답변
- `retrieved_docs`: 근거 청크 집합
- `tool_outputs`: 계산/날짜 파싱 결과 보정용

**실제 소스 코드: verify_grounding() — src/verifier.py**
```python
def verify_grounding(
    query: str,
    draft_answer: str,
    retrieved_docs: list[dict[str, Any]],
    tool_outputs: list[dict[str, Any]] | None = None,
) -> VerificationResult:
    if not retrieved_docs:
        return VerificationResult(
            is_grounded=False,
            coverage_score=0.0,
            unsupported_claims=["No retrieved evidence"],
            missing_aspects=[query],
        )

    evidence_token_sets = [content_tokens(doc["text"]) for doc in retrieved_docs]
    evidence_union = [token for token_set in evidence_token_sets for token in token_set]
    answer_sentences = [
        sentence
        for sentence in sentence_split(draft_answer)
        if not sentence.lower().startswith("sources:")
    ]

    calculator_tokens = {
        token
        for output in (tool_outputs or [])
        if output["tool_name"] == "calculator" and output["output"].get("ok")
        for token in content_tokens(str(output["output"]["result"]))
    }
    parsed_date_count = sum(
        1
        for output in (tool_outputs or [])
        if output["tool_name"] == "date_parser" and output["output"].get("ok")
    )

    unsupported_claims: list[str] = []
    support_scores: list[float] = []
    for sentence in answer_sentences:
        sentence_tokens = content_tokens(sentence)
        if not sentence_tokens:
            continue
        support = max(overlap_ratio(sentence_tokens, evidence_tokens) for evidence_tokens in evidence_token_sets)
        if calculator_tokens and calculator_tokens.issubset(set(sentence_tokens)) and parsed_date_count >= 2:
            support = max(support, 0.85)
        support_scores.append(support)
        if support < 0.45:
            unsupported_claims.append(sentence)

    query_coverage = overlap_ratio(content_tokens(query), evidence_union)
    answer_coverage = sum(support_scores) / len(support_scores) if support_scores else 0.0
    coverage_score = round(min(1.0, (answer_coverage * 0.75) + (query_coverage * 0.25)), 3)

    missing_aspects: list[str] = []
    if query_coverage < 0.35:
        missing_aspects.append("Evidence only covers part of the request.")

    is_grounded = not unsupported_claims and coverage_score >= 0.6
    return VerificationResult(
        is_grounded=is_grounded,
        coverage_score=coverage_score,
        unsupported_claims=unsupported_claims,
        missing_aspects=missing_aspects,
    )
```

**코드 읽기 포인트**
- `support < 0.45`인 문장은 unsupported claim으로 분류한다.
- `coverage_score`는 query coverage 25%, answer coverage 75%로 합성한다.
- `is_grounded = not unsupported_claims and coverage_score >= 0.6`: 문장별 unsupported가 없고 coverage가 충분해야 통과한다.
- 즉 verifier는 단순 점수 하나가 아니라, 왜 실패했는지 설명 가능한 구조(`unsupported_claims`, `missing_aspects`)를 만든다.

**결과 해석 가이드**
- `coverage_score`가 0.6 이상이고 `unsupported_claims`가 비어 있으면 대체로 grounded 답변으로 볼 수 있다.
- `missing_aspects`가 채워지면 근거는 일부 있지만 질문 전체를 덮지 못한 것이다.

**💡 면접 포인트**
- "Verifier는 answer quality보다 trustworthiness를 다루는 노드다. retrieval이 맞아도 unsupported claim이 있으면 실패로 본다"고 강조하면 좋다.


In [ ]:
verify_grounding_node(state)
pd.Series(state['verification_result'].to_dict())

## fallback 전략

검증 결과가 나왔다고 해서 항상 답을 내보내면 안 된다. 특히 문서 밖 질문이나 근거가 빈약한 질문은 보수적으로 abstain해야 한다. 이 단계가 없으면 verifier가 경고를 냈는데도 최종 시스템은 여전히 confident answer를 반환할 수 있다.

**목적**
- 답변할지, abstain할지 최종 의사결정을 내린다.

**핵심 로직**
- `insufficient_evidence_risk`는 특히 엄격하게 처리한다.
- grounded이고 coverage가 임계값 이상일 때만 답변을 확정한다.
- 그 외에는 근거 부족 메시지와 함께 `abstained`로 마무리한다.

**주요 파라미터/변수**
- `query_type`: 문서 밖 질문인지 여부
- `verification_result.coverage_score`: 답변 커버리지
- `verification_result.is_grounded`: 근거 충분성

**실제 소스 코드: fallback_or_finalize() — src/fallback.py**
```python
def fallback_or_finalize(
    query: str,
    query_type: str,
    draft_answer: str,
    verification_result: VerificationResult,
) -> dict[str, str]:
    if query_type == "insufficient_evidence_risk" and not (
        verification_result.is_grounded
        and verification_result.coverage_score >= 0.85
        and not verification_result.missing_aspects
    ):
        return {
            "final_answer": (
                "The loaded documents do not ground this request, so the workflow is abstaining instead of guessing."
            ),
            "final_status": "abstained",
        }

    if verification_result.is_grounded and verification_result.coverage_score >= 0.65:
        return {"final_answer": draft_answer, "final_status": "answered"}

    explanation = (
        "I do not have enough grounded evidence in the loaded documents to answer this confidently. "
        "Please narrow the question or add more source material."
    )
    if verification_result.missing_aspects:
        explanation = f"{explanation} Missing coverage: {'; '.join(verification_result.missing_aspects)}"

    return {"final_answer": explanation, "final_status": "abstained"}
```

**코드 읽기 포인트**
- `query_type == "insufficient_evidence_risk"`면 coverage가 높아 보여도 보수적으로 abstain할 수 있다.
- `coverage_score >= 0.65`는 최종 답변 허용 임계값이다.
- fallback은 "답을 못 했다"가 아니라 "근거가 충분하지 않아 추측을 거부했다"는 안전 장치다.

**결과 해석 가이드**
- `final_status`가 `abstained`라도 실패가 아니다. 문서 밖 질문이라면 오히려 바람직한 동작이다.


In [ ]:
fallback_or_finalize_node(state)
pd.Series({'final_status': state['final_status'], 'final_answer': state['final_answer']})

## workflow 실행

이제 개별 node가 아니라 전체 workflow를 돌려 본다. 아래 표는 query type별 대표 질문을 한 번씩 실행해, 분류 결과와 최종 상태가 어떻게 달라지는지 비교한다. 여기서 핵심은 "같은 엔진이 질문 성격에 따라 다른 경로를 탄다"는 점이다.

**목적**
- 9개 node가 하나의 end-to-end workflow로 어떻게 연결되는지 확인한다.

**핵심 로직**
- `run_workflow()`는 `WORKFLOW_STEPS`를 순서대로 실행한다.
- 현재 기본은 규칙 기반 synthesis/verifier이며, `use_llm=True`는 선택 사항이다.
- 각 node 실행 직후 `_record_node_trace()`와 `validate_state()`가 불려 상태 일관성을 점검한다.

**주요 파라미터/변수**
- `trace_steps`: 실행된 node 수. 정상 happy path면 9개 전후가 예상된다.
- `expected_demo_type`: notebook에서 의도한 시나리오 label
- `predicted_type`: classifier가 실제로 낸 결과

**실제 소스 코드: validate_state() — src/state_validation.py**
```python
NODE_REQUIREMENTS: dict[str, dict[str, type[Any]]] = {
    "normalize_query": {"normalized_query": str},
    "classify_query": {"query_type": str, "requires_tools": bool},
    "make_plan": {"plan": list},
    "retrieve_docs": {"retrieved_docs": list},
    "retrieve_memories": {"retrieved_memories": list},
    "decide_tools": {"tool_requests": list},
    "run_tools": {"tool_outputs": list},
    "synthesize_answer": {"draft_answer": str, "citations": list},
    "verify_grounding": {"verification_result": object},
    "fallback_or_finalize": {"final_answer": str, "final_status": str},
    "update_memory": {"memory_updates": list},
}


def _is_invalid(value: Any, expected_type: type[Any]) -> bool:
    if value is None:
        return True
    if expected_type is str:
        return not isinstance(value, str) or not value.strip()
    if expected_type is bool:
        return not isinstance(value, bool)
    if expected_type is list:
        return not isinstance(value, list)
    return False


def validate_state(state: AgentState) -> None:
    completed_nodes = {str(entry.get("node", "")) for entry in state.get("trace", [])}
    issues: list[str] = []

    for node_name, requirements in NODE_REQUIREMENTS.items():
        if node_name not in completed_nodes:
            continue
        for key, expected_type in requirements.items():
            if key not in state:
                issues.append(f"{key} missing after {node_name}")
                continue
            if _is_invalid(state[key], expected_type):
                issues.append(f"{key} invalid after {node_name}")

    if issues:
        issue_text = "; ".join(issues)
        raise StateValidationError(f"State validation failed: {issue_text}")
```

**코드 읽기 포인트**
- `NODE_REQUIREMENTS`는 node 완료 후 반드시 있어야 하는 키를 정의한다.
- trace에 어떤 node가 찍혔는지 보고, 그 node에 필요한 state가 빠졌다면 즉시 예외를 낸다.
- 이 검사가 있어야 "조용히 틀린 상태"가 아니라 "빨리 실패하는 상태"를 만들 수 있다.

**결과 해석 가이드**
- `predicted_type`과 `expected_demo_type`가 어긋나면 classifier를 먼저 본다.
- `trace_steps`가 유난히 적으면 중간에 예외나 조기 종료가 있었는지 trace를 확인한다.


In [ ]:
demo_queries = [
    ('simple_lookup', 'What are the main goals of the workspace policy refresh?'),
    ('comparison', 'How is the rollout plan different from the policy refresh?'),
    ('multi_hop', 'How many days are in the pilot window?'),
    ('summary', 'Summarize the loaded documents.'),
    ('insufficient_evidence_risk', 'Who is the current CEO of the company?'),
]
workflow_runs = []
for expected_type, question in demo_queries:
    result = run_workflow(question, retriever=retriever)
    workflow_runs.append(
        {
            'expected_demo_type': expected_type,
            'predicted_type': result['query_type'],
            'requires_tools': result['requires_tools'],
            'final_status': result['final_status'],
            'trace_steps': len(result['trace']),
            'question': question,
            'final_answer': result['final_answer'],
        }
    )

demo_frame = pd.DataFrame(workflow_runs)
demo_frame

## 실행 추적(trace) 살펴보기

trace는 workflow를 디버깅하는 가장 중요한 창이다. 표의 각 행은 하나의 node이고, `inputs`에는 그 node가 받은 값, `outputs`에는 그 node가 만든 값, `latency`에는 걸린 시간이 들어 있다. 즉 "어디서 무슨 판단이 일어났는가"를 시간 순서대로 재구성할 수 있다.

**목적**
- trace 테이블의 각 컬럼을 읽는 법을 익힌다.

**핵심 로직**
- `append_trace()`가 각 node entry에 `timestamp`, `inputs`, `outputs`, `latency`를 넣는다.
- `display_trace()`는 이를 DataFrame으로 풀어 notebook에서 읽기 좋게 보여준다.

**주요 파라미터/변수**
- `timestamp`: 노드가 기록된 시각
- `latency`: 해당 노드 실행 시간
- `inputs/outputs`: 디버깅 시 가장 먼저 볼 컬럼

**결과 해석 가이드**
- `normalize_query`의 output이 틀리면 초반 정규화 문제다.
- `retrieve_docs`의 outputs에 source가 이상하면 retrieval 문제다.
- `verify_grounding`에서 unsupported claim이 잡히면 synthesis는 자연스러워도 grounded answer는 아니다.

**💡 면접 포인트**
- "trace는 모델 사고를 흉내 낸 로그가 아니라, 실제 데이터 흐름과 상태 전이를 보여주는 운영 로그다"라고 설명하면 좋다.


In [ ]:
happy_path = run_workflow('How many days are in the pilot window?', retriever=retriever)
display_trace(happy_path['trace'])

## 실험

이제 happy path 하나만 보는 대신, 여러 query type의 결과를 한 표에 모아 비교한다. 이때 중요한 것은 정답 자체보다 `requires_tools`, `final_status`, `trace_steps`의 조합이다. 어떤 질문에서 도구가 켜졌는지, 어떤 질문에서 abstain했는지, 그 결정이 예상과 맞는지를 읽어야 workflow 품질을 설명할 수 있다.

**목적**
- 다양한 질문 유형에서 workflow의 분기 동작을 비교한다.

**결과 해석 가이드**
- `requires_tools=True`인데도 tool output이 없으면 tool planning 로직을 다시 봐야 한다.
- `insufficient_evidence_risk` 질문이 `answered`로 끝나면 fallback/verification 회귀를 의심해야 한다.


In [ ]:
demo_frame[['expected_demo_type', 'predicted_type', 'requires_tools', 'final_status', 'trace_steps', 'question']]

## 결과 해석

평균 `trace_steps`는 workflow가 질문 유형에 따라 얼마나 많은 reasoning 단계를 거쳤는지 보여주는 약식 지표다. 다만 steps가 많다고 무조건 좋은 것은 아니다. 핵심은 필요한 질문에 필요한 단계가 붙었는가, 그리고 최종 상태가 보수적으로 안전한가이다.

**목적**
- baseline과 agentic workflow의 차이를 구조적으로 읽는다.

**결과 해석 가이드**
- simple lookup은 적은 단계로 끝나도 괜찮지만, multi-hop과 insufficient-evidence 질문은 분기와 검증이 반드시 보이는 것이 좋다.
- agentic workflow의 가치는 "더 많은 일을 한다"가 아니라 "필요한 검증과 fallback을 한다"는 데 있다.


In [ ]:
demo_frame.groupby(['predicted_type', 'final_status'])['trace_steps'].mean().reset_index()

## 핵심 정리

이 노트북을 통해 9개 node가 하나의 stateful workflow로 연결되는 과정을 확인했다. `AgentState`는 단순 변수 묶음이 아니라, 분류 결과, 계획, 검색 근거, 도구 요청/출력, draft answer, verifier 판정, trace를 한 구조에서 관리하는 실행 계약이다. 각 node는 상태를 갱신하고 `_record_node_trace()`를 통해 trace와 latency를 남기며, `validate_state()`로 post-condition을 점검한다.

baseline과의 차이도 분명하다. baseline은 검색 후 바로 답변하지만, agentic workflow는 classify, plan, tool, verify, fallback을 추가해 복합 질문과 문서 밖 질문을 더 안전하게 다룬다. 특히 verifier와 fallback 덕분에 "모르면 모른다고 말하는" 동작을 설계 수준에서 강제할 수 있다.

**💡 면접 포인트**
- "Stateful workflow는 중간 상태를 명시적으로 보존해 디버깅과 평가를 쉽게 만든다."
- "Trace의 inputs/outputs/latency를 보면 어느 node가 병목인지, 어느 node가 잘못된 결정을 했는지 빠르게 찾을 수 있다."
- "LLM 경로(`use_llm=True`)를 추가하더라도, 기본 규칙 기반 workflow와 동일한 state contract를 유지하면 안전하게 확장할 수 있다. 자세한 내용은 09번 notebook에서 이어진다."
